# IMPORT LIBRARIES

In [1]:
from optimization_model import build_model
import pyomo.environ as pyo
from __future__ import annotations
import json
import os

In [2]:
def load_instance_json(path: str) -> Dict[str, Any]:
    """Load instance from JSON. Restores tuple keys for known fields."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # Fields with tuple keys and their arity
    tuple_key_fields = {
        "dur": 2,     # (t,k)
        "dist": 2,    # (i,j)
        "tt": 3,      # (i,j,m)
        "Hkp": 2,     # (k,p)
    }

    def decode_keys(d: Dict[str, Any], field_name: str) -> Dict[Any, Any]:
        arity = tuple_key_fields.get(field_name, None)
        if arity is None:
            return d
        out = {}
        for k, v in d.items():
            parts = k.split("|")
            if len(parts) != arity:
                # leave as string key if unexpected
                out[k] = v
            else:
                out[tuple(parts)] = v
        return out

    data: Dict[str, Any] = dict(raw)
    for fn in tuple_key_fields:
        if fn in data and isinstance(data[fn], dict):
            data[fn] = decode_keys(data[fn], fn)

    return data

In [3]:
def export_results(model: pyo.ConcreteModel, filepath: str, save_zero: bool = False):
    """
    Export:
      - objective value
      - all variable values
    to a JSON file.

    save_zero = False -> only store non-zero variables (recommended)
    """

    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)

    results = {}

    # ------------------------
    # Objective
    # ------------------------
    obj = next(model.component_data_objects(pyo.Objective, active=True))
    results["objective_value"] = float(pyo.value(obj))

    # ------------------------
    # Variables
    # ------------------------
    results["variables"] = {}

    for var in model.component_objects(pyo.Var, active=True):
        var_name = var.name
        results["variables"][var_name] = {}

        for index in var:
            val = pyo.value(var[index])
            if (not save_zero) and (abs(val) < 1e-9):
                continue

            # Convert index to string
            if isinstance(index, tuple):
                key = "|".join(map(str, index))
            else:
                key = str(index)

            results["variables"][var_name][key] = float(val)

    # ------------------------
    # Save JSON
    # ------------------------
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {filepath}")

# IMPORT DATA

In [4]:
data = load_instance_json("instances\demo_clustered_seed1.json")
data

{'B': ['b1', 'b2'],
 'K': ['k1', 'k2', 'k3', 'k4'],
 'P': ['p1', 'p2', 'p3', 'p4', 'p5', 'p6'],
 'T': ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 't10'],
 'D': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30],
 'M': ['car', 'train', 'air'],
 'N': ['b1', 'b2', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6'],
 'a': {'t1': 8,
  't2': 1,
  't3': 18,
  't4': 10,
  't5': 11,
  't6': 17,
  't7': 10,
  't8': 17,
  't9': 2,
  't10': 13},
 'b': {'t1': 17,
  't2': 9,
  't3': 21,
  't4': 13,
  't5': 21,
  't6': 22,
  't7': 19,
  't8': 25,
  't9': 11,
  't10': 21},
 'w': {'t1': 2.0,
  't2': 5.0,
  't3': 1.0,
  't4': 4.0,
  't5': 2.0,
  't6': 2.0,
  't7': 5.0,
  't8': 3.0,
  't9': 1.0,
  't10': 4.0},
 'dur': {('t1', 'k1'): 2,
  ('t1', 'k2'): 3,
  ('t1', 'k3'): 2,
  ('t1', 'k4'): 2,
  ('t2', 'k1'): 999,
  ('t2', 'k2'): 2,
  ('t2', 'k3'): 1,
  ('t2', 'k4'): 2,
  ('t3', 'k1'): 1,
  ('

# RUN

In [5]:
m = build_model(data)
solver = pyo.SolverFactory("cplex")  # or "cbc"/"glpk" depending on your environment
solver.options["timelimit"] = 300   # 5 minuti = 300 secondi
res = solver.solve(m, tee=True)
export_results(m, "results/demo_clustered_seed1.json")


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmp14dlh5_w.cplex.log' open.
CPLEX> New value for time limit in seconds: 300
CPLEX> Problem 'C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpqxcx0_od.pyomo.lp' read.
Read time = 0.09 sec. (5.68 ticks)
CPLEX> Problem name         : C:\Users\MATTEO~1.GAB\AppData\Local\Temp\tmpqxcx0_od.pyomo.lp
Objective sense      : Minimize
Variables            :   26780  [Nneg: 20,  Binary: 26760]
Objective nonzeros   :   21490
Linear constraints   :   38061  [Less: 27910,  Greater: 10,  Equal: 10141]
  Nonzeros           :  236056
  RHS nonzeros       :     604

Variables 